# 📝 ReAct·멀티툴 에이전트 과제 LV1 정답 (강사용)

각 문제의 **모범답안 + 해설**입니다. 학생이 스스로 푼 뒤 비교하도록 안내하세요.

- 경로는 정답 노트북 기준 `../../day20_ReAct_멀티툴_에이전트/data/` 입니다.
- 순수 함수(조각 번호 고르기) 문제와 도구 자체는 **값을 정확히** 검사합니다(모델과 무관하게 결정적).
- 반면 **모델이 만든 기록**은 실제 호출 결과라 매번 조금씩 달라집니다(스텝 수·도구 호출 횟수·순서·문장). 그래서 기록 문제의 자가채점은 **구조**(리스트인지, 그 도구가 최소 1회 불렸는지)만 보고, 라우팅은 **출력으로 확인**하도록 했습니다.

In [ ]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../../day20_ReAct_멀티툴_에이전트/.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 - 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 오늘 쓸 모델. 18일차에서 배운 그대로입니다(이 셀은 실행만 하세요).
from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

In [ ]:
# [제공 코드] 구청 민원 수수료표·처리기한표를 불러옵니다(이 셀은 실행만 하세요)
import re
import pandas as pd
_fees = pd.read_csv('../../day20_ReAct_멀티툴_에이전트/data/fees.csv')
FEE_TABLE = dict(zip(_fees['document'], _fees['fee']))       # 예: {'주민등록등본': 400, ...}
_dls = pd.read_csv('../../day20_ReAct_멀티툴_에이전트/data/deadlines.csv')
DEADLINE_TABLE = dict(zip(_dls['minwon'], _dls['days']))     # 예: {'여권 발급': 8, ...}
print('수수료 항목:', list(FEE_TABLE)[:3], '...')
print('처리기한 항목:', list(DEADLINE_TABLE)[:3], '...')

## 1. 외부 REST API 를 도구로 - 공휴일 조회
**배경**: 민원 처리 기한을 안내하려면 **공휴일**을 알아야 하는데, 모델은 그것을 모릅니다. 인터넷의 공휴일 서비스를 **도구로 감싸** 쥐여 줍니다. 키가 필요 없는 공개 서비스입니다.

```text
https://date.nager.at/api/v3/PublicHolidays/{연도}/KR
```

응답은 `[{'date': '2026-01-01', 'localName': '새해', ...}, ...]` 형태의 목록입니다.

**요구사항**:
- `@tool` 로 **`holiday_names(year: int) -> str`** 를 만드세요. 그 해 한국 공휴일의 **이름(`localName`)을 쉼표로 이어** 한 문자열로 돌려줍니다.
- docstring 첫 줄에 **무엇을 하는 도구인지**를 적으세요(모델이 읽는 설명서입니다).
- 요청은 **`timeout=10`** 으로 보내고, **실패는 예외로 던지지 말고 문자열로** 돌려주세요 - `except requests.RequestException` 으로 잡아 안내 문장을 반환합니다.

**예시**: `holiday_names.invoke({'year': 2026})` → `'새해, 설날, 설날, ...'`

<details><summary>힌트</summary>

```text
접근방법:
- 요청을 try 안에 두고, 성공하면 응답 목록에서 이름만 뽑아 잇는다.

세부구현:
1. @tool 을 붙인 함수를 정의하고 타입힌트와 docstring 을 적는다.
2. try 안에서 requests 로 주소를 부르고, 응답 상태를 확인한다.
3. RequestException 을 잡으면 안내 문장을 문자열로 반환한다.
4. 성공하면 응답의 각 항목에서 localName 만 모아 쉼표로 이어 반환한다.
```

</details>

In [ ]:
# [제공 코드] 이 문제에 필요한 import - 이 셀은 실행만 하세요.
import requests
from langchain_core.tools import tool
print('준비 완료')

In [ ]:
# 실패를 문자열로 돌려주는 것이 핵심 - 예외를 던지면 에이전트가 그 자리에서 멈춘다
@tool
def holiday_names(year: int) -> str:
    """그 해 한국 공휴일의 이름을 모두 알려준다. 쉬는 날인지 확인할 때 쓴다."""
    try:
        res = requests.get(f'https://date.nager.at/api/v3/PublicHolidays/{year}/KR',
                           timeout=10)
        res.raise_for_status()
    except requests.RequestException as error:
        return f'공휴일을 가져오지 못했습니다: {error}. 잠시 뒤 다시 시도해 주세요.'
    return ', '.join(h['localName'] for h in res.json())

print(holiday_names.invoke({'year': 2026}))

In [ ]:
# [자가채점] - 인터넷이 끊겨도 '문자열을 돌려주는지' 를 봅니다(그것이 이 문제의 요점입니다).
from langchain_core.tools import BaseTool
assert isinstance(holiday_names, BaseTool), '@tool 을 붙였는지 확인하세요'
assert holiday_names.description.strip(), 'docstring 을 적었는지 확인하세요'
_out = holiday_names.invoke({'year': 2026})
assert isinstance(_out, str) and _out.strip(), '문자열을 돌려줘야 합니다'

# 네트워크가 끊긴 상황을 흉내 내 봅니다 - 도구는 멈추지 말고 문자열로 알려 줘야 합니다.
_real_get = requests.get


def _broken_get(*args, **kwargs):
    raise requests.RequestException('테스트용 네트워크 끊김')


requests.get = _broken_get
try:
    _fallback = holiday_names.invoke({'year': 2026})
except Exception as error:
    raise AssertionError('실패를 예외로 던졌습니다 - 문자열로 돌려주세요') from error
finally:
    requests.get = _real_get
assert isinstance(_fallback, str) and _fallback.strip()
print('✅ 통과!')

**해설**: 도구 두 가지를 함께 훈련하는 문제입니다. **1)** `@tool` 과 docstring — 모델은 이 설명만 보고 도구를 고릅니다. **2)** **실패도 문자열로** — 인터넷은 언제든 끊기는데, 예외를 밖으로 던지면 에이전트가 그 자리에서 멈춥니다. 문자열로 돌려주면 모델이 그것을 관찰로 읽고 사용자에게 안내하거나 다시 시도합니다.

**흔한 실수**: `raise_for_status()` 를 `try` 밖에 두는 것입니다. 4xx·5xx 응답에서 예외가 그대로 밖으로 나갑니다.

---
## 2. 메시지 기록에서 도구 호출 이름·id 뽑기
**배경**: `create_agent` 는 텍스트 파싱 없이 **구조화된 `tool_calls`** 로 행동을 남깁니다. 위 준비 셀이 만들어 둔 `result` 의 기록에서 **어떤 도구가 불렸는지**를 읽어 봅니다.

**요구사항**:
- `result['messages']` 를 돌며, 각 메시지의 `tool_calls` 에서 `call['name']` 을 모아 리스트 **`called_names`** 에, `call['id']` 를 모아 리스트 **`called_ids`** 에 담으세요. `tool_calls` 는 **`AIMessage` 에만** 있습니다 — 다른 메시지에 그대로 쓰면 에러가 나므로 `isinstance(message, AIMessage)` 로 먼저 걸러 내세요(`AIMessage` 는 준비 셀이 임포트해 둡니다).
- `id` 는 **그 호출을 가리키는 번호표**입니다. 부를 때마다 새로 생기는 값이라 기록에서 읽어야만 알 수 있고, 나중에 도구 결과(`ToolMessage`)를 어느 호출의 답인지 잇는 데 쓰입니다.

**예시**: 준비 셀은 수수료 질문을 넣었으므로 `called_names` 에 **`'lookup_fee'` 가 들어갑니다** (모델이 도구를 두 번 부르는 등 개수는 실행마다 달라질 수 있어, 이름이 **들어 있는지**만 봅니다). `called_ids` 는 `['call_...']` 처럼 실행마다 다른 문자열이 담깁니다.

<details><summary>힌트</summary>

```text
접근방법:
- 메시지를 돌며 tool_calls 가 있으면 그 안의 각 호출에서 name 과 id 를 모은다.

세부구현:
1. 빈 리스트 called_names 와 called_ids 를 만든다.
2. result['messages'] 를 돌며 그 메시지가 AIMessage 인지 먼저 확인한다.
3. 맞으면 그 메시지의 tool_calls 를 돌며 name 은 called_names 에, id 는 called_ids 에 추가한다.
```

</details>

In [ ]:
# [제공 코드] 기록을 만들어 둡니다 - 아래 result 를 2~4번에서 읽습니다(이 셀은 실행만 하세요)
from langchain_core.tools import tool
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain.agents import create_agent

@tool
def lookup_fee(document: str, count: int) -> str:
    """민원 서류 이름과 매수를 받아 총 발급 수수료(원)를 돌려준다."""
    doc = document.strip()
    return str(FEE_TABLE.get(doc, 0) * count)

_demo_agent = create_agent(model, [lookup_fee],
                           system_prompt="너는 구청 민원 도우미다. 수수료는 반드시 lookup_fee 도구로 계산해 답하라.")
result = _demo_agent.invoke({"messages": [HumanMessage("주민등록등본 2장 수수료 알려줘")]})

# 4번에서 같은 에이전트에 다른 질문을 넣어 봅니다.
agent = _demo_agent
calc_fee_tool = lookup_fee

print("기록 메시지 수:", len(result["messages"]))

In [ ]:
# tool_calls 는 도구를 부른 AIMessage 에만 있다 - 다른 메시지에 그대로 쓰면 에러
called_names = []
called_ids = []
for message in result['messages']:
    if isinstance(message, AIMessage):
        for call in message.tool_calls:
            called_names.append(call['name'])
            called_ids.append(call['id'])
print(called_names, called_ids)

In [ ]:
# [자가채점] - 기록은 실제 호출 결과라 도구 호출 '횟수·순서'는 매번 달라질 수 있습니다.
# 그래서 이름 목록의 정확한 모양은 보지 않고, 수수료 도구가 들어 있는지만 봅니다.
assert isinstance(called_names, list) and isinstance(called_ids, list)
assert 'lookup_fee' in called_names   # 수수료 질문이라 수수료 도구가 불렸다
# id 는 부를 때마다 새로 생기는 값이라 기록에서 읽어야만 알 수 있습니다 -
# 손으로 적은 목록은 여기서 걸립니다(값도, 개수도 맞출 수 없습니다).
traj_ids = []
for message in result['messages']:
    if isinstance(message, AIMessage):
        for call in message.tool_calls:
            traj_ids.append(call['id'])
assert called_ids == traj_ids
assert len(called_names) == len(traj_ids)
print('✅ 통과!')

**해설**: `tool_calls` 는 도구를 부른 `AIMessage` 에만 있습니다. `isinstance` 로 그 메시지만 골라 낸 뒤 각 호출의 `name` 을 모으면 **에이전트가 실제로 어떤 도구를 골랐는지** 구조적으로 확인할 수 있습니다(문장을 읽을 필요가 없습니다).

자가채점이 `== ['lookup_fee']` 가 아니라 `in` 인 이유: 모델을 **실제로** 부르기 때문에 같은 질문이라도 도구를 한 번 부르고 끝낼 수도, 확인차 한 번 더 부를 수도 있습니다. **호출 횟수·순서는 모델이 정하는 일**이라 채점 대상이 아니고, 학생이 검증받아야 하는 것은 **기록에서 이름을 뽑아내는 코드**입니다.

그래서 `called_ids` 를 함께 모읍니다. `id` 는 **호출할 때마다 새로 생기는 값**이라 실행해 보지 않고는 알 수 없습니다 — 결과를 손으로 적어 넣으면 이름은 맞출 수 있어도 id 는 맞출 수 없어 자가채점이 걸립니다. 이 `id` 는 8번의 제공 셀 `one_beat` 에서 도구 결과를 `ToolMessage(..., tool_call_id=...)` 로 되먹일 때 **어느 호출의 답인지 잇는 열쇠**로 다시 만납니다.

---
## 3. 모델이 몇 번 말했고 도구가 무엇을 돌려줬나
**배경**: 2번에서 **무엇을 불렀는지**를 읽었습니다. 이번에는 같은 기록에서 **몇 모델이 말했는지**와 **도구가 실제로 돌려준 값**을 뽑습니다. 에이전트를 고칠 때마다 이 두 수가 어떻게 변하는지 보게 됩니다.

**요구사항**: 준비 셀이 만들어 둔 `result` 를 씁니다.

- `result['messages']` 에서 **`AIMessage` 의 개수**를 세어 **`n_steps`** 에 담으세요(정수). 모델이 말한 횟수가 곧 루프가 돈 모델이 말한 횟수입니다.
- `result['messages']` 에서 **`ToolMessage` 의 `content`** 를 모아 리스트 **`observations`** 에 담으세요. 이것이 ReAct 의 **관찰**입니다.
- 두 값을 출력하세요.

**예시**: 수수료 질문이라 `observations` 에는 계산 결과 문자열이 들어 있고, `n_steps` 는 2 이상입니다 (도구를 부르는 단계 + 답하는 단계).

<details><summary>힌트</summary>

```text
접근방법:
- 메시지 종류로 걸러 내면 된다. isinstance 로 종류를 확인한다.

세부구현:
1. 메시지를 돌며 AIMessage 인 것의 개수를 센다.
2. 메시지를 돌며 ToolMessage 인 것의 content 를 모은다.
3. 두 값을 출력한다.
```

</details>

In [ ]:
# AIMessage 는 '모델이 말한 차례' 다 - 도구를 부른 차례와 최종답을 낸 차례가 모두 여기 해당한다.
n_steps = sum(1 for m in result['messages'] if isinstance(m, AIMessage))
observations = [m.content for m in result['messages'] if isinstance(m, ToolMessage)]

print('모델이 말한 횟수:', n_steps)
print('관찰   :', observations)

In [ ]:
# [자가채점] - 손으로 적은 값이 아니라 기록을 실제로 센 값인지 대조합니다.
assert isinstance(n_steps, int) and n_steps >= 2, '도구를 부른 단계와 답한 단계가 있어야 합니다'
assert n_steps == sum(1 for m in result['messages'] if isinstance(m, AIMessage))
assert isinstance(observations, list) and observations, '관찰이 비어 있습니다'
assert observations == [m.content for m in result['messages'] if isinstance(m, ToolMessage)]
print('✅ 통과!')

**해설**: **모델이 말한 횟수를 정하는 것은 질문**입니다. 도구가 필요 없으면 `AIMessage` 하나로 끝나고, 도구를 부르면 최소 둘(부르는 차례 + 결과를 보고 답하는 차례)이 됩니다. 도구 결과를 보고 또 다른 도구를 불러야 하면 더 늘어납니다.

**`ToolMessage.content` 가 관찰입니다.** ReAct 의 세 단계에서 '관찰' 에 해당하는 것이 바로 이 값이고, 모델은 이것을 읽고 다음 행동을 정합니다. 그래서 도구가 무엇을 돌려주는지가 곧 **모델에게 주는 정보의 질**입니다.

---
## 4. 도구가 필요 없는 질문은 기록이 어떻게 다른가
**배경**: 에이전트의 미덕은 **필요할 때만 도구를 쓰는 것**입니다. 도구는 그대로 붙여 두고 **질문만** 바꿔 기록이 어떻게 달라지는지 봅니다(2·3번과 같은 에이전트, 다른 질문).

**요구사항**:
- 준비 셀의 `agent` 에 **"구청 민원 창구는 무슨 일을 하나요? 한 문장으로 알려줘."** 를 `invoke` 한 결과를 **`res_notool`** 에 담으세요.
- 그 기록에서 **`ToolMessage` 의 개수**를 세어 **`n_tool_msgs`** 에 담으세요(정수).
- 최종 답 문자열을 **`plain_answer`** 에 담으세요(`res_notool['messages'][-1].text`).
- 두 값을 출력하세요.

**예시**: `n_tool_msgs` 는 **0** 이고 `plain_answer` 는 비어 있지 않은 문장입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 도구를 부를 이유가 없는 질문을 넣고, 기록에 도구 결과가 하나도 없는지 센다.

세부구현:
1. 지문의 질문을 그대로 넣어 invoke 하고 결과를 담는다.
2. 기록에서 도구 결과 메시지의 개수를 센다.
3. 마지막 메시지의 글을 꺼낸다.
```

</details>

In [ ]:
res_notool = agent.invoke(
    {'messages': '구청 민원 창구는 무슨 일을 하나요? 한 문장으로 알려줘.'})

# 도구는 그대로 붙어 있다 - 부를 이유가 없다고 모델이 판단한 것이다.
n_tool_msgs = sum(1 for m in res_notool['messages'] if isinstance(m, ToolMessage))
plain_answer = res_notool['messages'][-1].text

print('도구 결과 개수:', n_tool_msgs)
print('최종 답      :', plain_answer)

In [ ]:
# [자가채점]
# 0 은 손으로도 적을 수 있는 값이라, res_notool 을 실제로 실행했는지부터 확인합니다.
assert res_notool['messages'][0].text.strip().startswith('구청 민원 창구는'), \
    '지문의 질문을 그대로 넣어 실행하세요'
assert n_tool_msgs == sum(1 for m in res_notool['messages'] if isinstance(m, ToolMessage))
assert n_tool_msgs == 0, '도구가 필요 없는 질문인데 도구가 불렸습니다'
assert isinstance(plain_answer, str) and plain_answer.strip()
print('✅ 통과!')

**해설**: 2·3번과 **같은 에이전트, 같은 도구**입니다. 달라진 것은 질문뿐이지요. 그런데 기록에 `ToolMessage` 가 하나도 없습니다 — 모델이 "이건 도구 없이 답할 수 있다" 고 판단한 것입니다.

**이것이 체인과 에이전트의 결정적 차이**입니다. 체인이었다면 경로가 고정이라 도구(또는 검색)를 반드시 지나갔을 것입니다. 판단을 모델에게 넘긴 대가로 얻는 것이 이 유연함입니다.

## 5. 좋은 docstring 으로 수수료 도구 만들기
**배경**: `@tool` 의 **docstring 이 곧 도구 설명**입니다. 에이전트는 이 설명을 읽고 도구를 고르므로, 설명이 분명해야 알맞은 질문에 이 도구를 부릅니다.

**요구사항**:
- `@tool` 로 함수 **`calc_fee(document: str, count: int)`** 를 만드세요(타입 힌트까지 그대로). `document`(서류 이름)와 `count`(매수)를 받아 **`FEE_TABLE` 에서 단가를 찾아 매수를 곱한 값**을 문자열로 돌려줍니다. 표에 없으면 `'0'`.
- docstring 에 **무엇을 하는 도구인지**(민원 서류 발급 수수료 계산)를 한국어로 **열다섯 자 이상**, **`수수료`** 라는 말이 들어가게 적으세요(자가채점이 이 두 가지를 봅니다).
- 입력 표면형이 흔들릴 수 있으니 `document` 는 **`.strip()`** 으로 다듬어 조회하세요.
- 만든 도구로 에이전트를 만들어(`fee_agent`) **'가족관계증명서 2장 발급 수수료는?'** 을 물어, 그 결과를 **`fee_result`** 에 담으세요.

**예시**: 이 질문에는 `calc_fee` 도구가 불려야 합니다(기록에 `calc_fee` 가 나타남).

<details><summary>힌트</summary>

```text
접근방법:
- @tool 로 함수를 감싸고 docstring 을 분명히 쓴다. 에이전트를 만들어 invoke 한다.

세부구현:
1. @tool 데코레이터를 붙여 calc_fee(document, count) 를 정의한다(타입 힌트 str·int).
2. docstring 에 '민원 서류 발급 수수료를 계산한다' 처럼 용도를 적는다.
3. 함수 안에서 document 를 strip 하고 FEE_TABLE.get(그것, 0) * count 를 문자열로 반환한다.
4. create_agent(model, [calc_fee], system_prompt=...) 로 fee_agent 를 만든다.
5. fee_agent.invoke 로 예시 질문을 넣어 fee_result 에 담는다.
```

</details>

In [ ]:
# docstring 이 곧 도구 설명이다 — 비워 두면 에이전트가 언제 쓸지 몰라 라우팅이 흔들린다
@tool
def calc_fee(document: str, count: int) -> str:
    """민원 서류 이름과 매수를 받아 총 발급 수수료(원)를 계산한다."""
    # 표에 없는 서류는 0 원 — .get 의 기본값으로 KeyError 를 막는다
    return str(FEE_TABLE.get(document.strip(), 0) * count)

fee_agent = create_agent(model, [calc_fee],
                         system_prompt='너는 구청 민원 도우미다. 필요하면 도구를 불러 답하라.')
fee_result = fee_agent.invoke({'messages': [HumanMessage('가족관계증명서 2장 발급 수수료는?')]})
print('불린 도구:', [call['name'] for message in fee_result['messages']
                    if isinstance(message, AIMessage)
                    for call in message.tool_calls])   # ['calc_fee']
print(fee_result['messages'][-1].text)

In [ ]:
# [자가채점] - 도구 자체(이름·계산값·설명)를 결정적으로 검사합니다.
# 에이전트가 calc_fee 를 부르는지(라우팅)는 위 답안 셀의 '불린 도구:' 출력으로 눈으로 확인하세요.
from langchain_core.tools import BaseTool
assert isinstance(calc_fee, BaseTool)   # @tool 로 만든 진짜 도구인가(그냥 함수·흉내 객체면 여기서 걸림)
assert calc_fee.name == 'calc_fee'
# docstring 은 '있기만' 하면 안 됩니다 - 무엇을 하는 도구인지가 드러나야 에이전트가 고를 수 있습니다.
assert len(calc_fee.description) >= 15, 'docstring 을 열다섯 자 이상으로 분명히 적으세요'
assert '수수료' in calc_fee.description, "docstring 에 '수수료' 가 들어가야 합니다"
assert calc_fee.invoke({'document': '가족관계증명서', 'count': 2}) == '2000'   # 도구 자체는 결정적
# 에이전트를 실제로 돌렸는지 - 기록에 질문과 '모델이 만든 AIMessage' 가 함께 있어야 합니다.
assert any(m.content == '가족관계증명서 2장 발급 수수료는?' for m in fee_result['messages'])
assert any(isinstance(m, AIMessage) and m.response_metadata
           for m in fee_result['messages'])
print('✅ 통과!')

**해설**: 도구 자체(`calc_fee.invoke`)는 `FEE_TABLE` 조회라 **결정적**이라 값(2000)을 정확히 검사할 수 있습니다. 반면 에이전트가 이 도구를 고르는지(라우팅)는 **모델이 매 호출마다 정하는 일**이라 자가채점하지 않고, 답안 셀의 `불린 도구:` 출력으로 **눈으로 확인**합니다 — 보통 `['calc_fee']` 가 찍힙니다. 흔한 실수: docstring 을 비워 두면 에이전트가 도구 용도를 몰라 엉뚱하게 답할 수 있습니다(그래서 설명 길이도 검사합니다).

## 6. 좋은 docstring 으로 처리기한 도구 만들기
**배경**: 5번과 같은 방식으로, 이번엔 **처리 기한 안내** 도구를 만듭니다(같은 개념, 다른 도구·표).

**요구사항**:
- `@tool` 로 함수 **`lookup_deadline(minwon)`** 를 만드세요. 민원 이름을 받아 `DEADLINE_TABLE` 에서 **처리 일수**를 찾아 `'N일'` 형태 문자열로 돌려줍니다(예: `'8일'`). 표에 없으면 `'모름'`.
- docstring 에 **처리 기한(며칠 걸리는지)을 알려 주는 도구**임을 **열다섯 자 이상**으로 적으세요 — **`기한`·`기간`·`며칠`** 중 하나는 반드시 들어가야 합니다. `minwon` 은 `.strip()`.
- 만든 도구로 에이전트(`deadline_agent`)를 만들어 **'여권 발급은 처리에 며칠 걸리나요?'** 를 물어, 그 결과를 **`dl_result`** 에 담으세요.

**예시**: 이 질문에는 `lookup_deadline` 도구가 불려야 합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 5번과 같은 틀. 반환 형식만 'N일' 문자열로 바꾼다.

세부구현:
1. @tool 로 lookup_deadline(minwon) 을 정의하고 docstring 을 분명히 쓴다.
2. DEADLINE_TABLE.get(minwon.strip()) 로 일수를 찾는다.
3. 있으면 f'{일수}일', 없으면 '모름' 을 반환한다.
4. create_agent(model, ...) 로 deadline_agent 를 만들고 예시 질문을 invoke 해 dl_result 에 담는다.
```

</details>

In [ ]:
# 5번과 같은 틀 — 다른 표를 쓰고 반환 형식만 'N일' 문자열로 바꾼다
@tool
def lookup_deadline(minwon: str) -> str:
    """민원 이름을 받아 접수 후 처리에 걸리는 기간(며칠)을 알려 준다."""
    days = DEADLINE_TABLE.get(minwon.strip())
    # 0 일인 민원이 생겨도 안전하도록 is not None 으로 검사한다(0 은 거짓이라 if days 면 새어 나간다)
    return f'{days}일' if days is not None else '모름'

deadline_agent = create_agent(model, [lookup_deadline],
                              system_prompt='너는 구청 민원 도우미다. 필요하면 도구를 불러 답하라.')
dl_result = deadline_agent.invoke({'messages': [HumanMessage('여권 발급은 처리에 며칠 걸리나요?')]})
print('불린 도구:', [call['name'] for message in dl_result['messages']
                    if isinstance(message, AIMessage)
                    for call in message.tool_calls])   # ['lookup_deadline']
print(dl_result['messages'][-1].text)

In [ ]:
# [자가채점] - 도구 자체(이름·값·설명)를 검사. 라우팅은 위 '불린 도구:' 출력으로 눈으로 확인.
from langchain_core.tools import BaseTool
assert isinstance(lookup_deadline, BaseTool)   # @tool 로 만든 진짜 도구인가
assert lookup_deadline.name == 'lookup_deadline'
assert len(lookup_deadline.description) >= 15, 'docstring 을 열다섯 자 이상으로 분명히 적으세요'
assert any(w in lookup_deadline.description for w in ['기한', '기간', '며칠']), \
    "docstring 에 '기한'·'기간'·'며칠' 중 하나가 들어가야 합니다"
assert lookup_deadline.invoke({'minwon': '여권 발급'}) == '8일'   # 도구 자체는 결정적
# 에이전트를 실제로 돌렸는지 - 기록에 질문과 '모델이 만든 AIMessage' 가 함께 있어야 합니다.
assert any(m.content == '여권 발급은 처리에 며칠 걸리나요?' for m in dl_result['messages'])
assert any(isinstance(m, AIMessage) and m.response_metadata
           for m in dl_result['messages'])
print('✅ 통과!')

**해설**: 5번과 **같은 개념**을 다른 도구·표로 한 번 더 훈련했습니다. 도구 값(`'8일'`)과 설명 유무는 결정적으로 검사하고, 에이전트가 이 도구를 부르는 라우팅은 답안 셀의 `불린 도구:` 출력으로 확인합니다.

---
## 7. 두루뭉술한 도구 설명 고치기
**배경**: 교안 5절에서 **이름과 설명만** 바꿔도 에이전트의 선택이 달라지는 것을 봤습니다. 아래 제공 셀의 `find` 는 **하는 일은 맞는데** 이름도 설명도 두루뭉술해, 모델 입장에서는 언제 부를 도구인지 알 길이 없습니다. 같은 일을 하는 도구를 **설명만 제대로 붙여** 다시 만듭니다.

**요구사항**:
- `@tool` 로 함수 **`find_window(minwon: str) -> str`** 를 만드세요. `_WINDOW` 에서 담당 창구를 찾아 돌려줍니다(`minwon` 은 `.strip()`).
- 표에 없는 민원이면 `'없음'` 대신 **안내 문자열**을 돌려주세요 — 그 문장 안에 고를 수 있는 민원 이름 **`여권`·`전입신고`·`등본 발급`** 이 들어가야 합니다.
- docstring 에는 두 가지를 적으세요. (1) **언제 쓰는 도구인지** — **`창구`** 라는 말이 들어가게, (2) **고를 수 있는 민원 이름 세 개**(`여권`·`전입신고`·`등본 발급`) 그대로.
- 만든 도구로 에이전트 **`window_agent`** 를 만들어 **'여권 발급하려면 어디로 가야 하나요?'** 를 물어 결과를 **`win_result`** 에 담고, 불린 도구를 출력하세요.

**예시**: `find_window('여권')` 은 `'3번 창구'` 이고, `find_window('없는민원')` 은 세 민원 이름이 들어간 안내 문장입니다. 이 질문에는 `find_window` 가 불립니다.

<details><summary>힌트</summary>

```text
접근방법:
- 도구를 고르는 근거는 오직 이름과 설명뿐이다. '무엇을 알려 주는지' 와 '무엇을 넣을 수 있는지' 를
  설명에 적어 준다.

세부구현:
1. @tool 로 find_window(minwon) 을 정의한다.
   1-1. docstring 첫 줄에 담당 창구를 알려 주는 도구임을 적는다.
   1-2. 다음 줄에 고를 수 있는 민원 이름 세 개를 적는다.
2. 민원 이름의 앞뒤 공백을 떼고 표에서 찾는다.
3. 없으면 세 민원 이름을 알려 주는 안내 문장을 돌려준다.
4. create_agent 로 에이전트를 만들어 예시 질문을 invoke 하고 불린 도구를 출력한다.
```

</details>

In [ ]:
# [제공 코드] 담당 창구표와, 설명이 두루뭉술한 도구 - 이 셀은 실행만 하세요.
_WINDOW = {'여권': '3번 창구', '전입신고': '1번 창구', '등본 발급': '무인발급기'}


@tool
def find(x: str) -> str:
    """찾는다."""
    return _WINDOW.get(x, '없음')


# 모델이 보는 것은 이 두 줄이 전부입니다 - 이것만 보고 '언제 쓸 도구인지' 알 수 있나요?
print('이름:', find.name)
print('설명:', find.description)

In [ ]:
# 이름과 설명이 곧 모델이 읽는 사용설명서다 — 무엇을 알려 주는지, 무엇을 넣을 수 있는지를 적는다
@tool
def find_window(minwon: str) -> str:
    """민원 이름을 받아 그 민원을 처리하는 담당 창구를 알려 준다.

    민원 이름은 여권·전입신고·등본 발급 중 하나다.
    """
    name = minwon.strip()
    # 없는 값에 '없음' 을 돌려주면 모델이 '정말 없다' 로 읽는다 - 무엇을 고를지 알려 준다
    if name not in _WINDOW:
        return f"'{name}' 은 안내 목록에 없습니다. 여권·전입신고·등본 발급 중에서 고르세요."
    return _WINDOW[name]

window_agent = create_agent(model, [find_window],
                            system_prompt='너는 구청 민원 도우미다. 필요하면 도구를 불러 답하라.')
win_result = window_agent.invoke(
    {'messages': [HumanMessage('여권 발급하려면 어디로 가야 하나요?')]})
print('불린 도구:', [call['name'] for message in win_result['messages']
                    if isinstance(message, AIMessage)
                    for call in message.tool_calls])   # ['find_window']
print(win_result['messages'][-1].text)

In [ ]:
# [자가채점] - 설명(docstring)에 무엇이 담겼는지를 결정적으로 검사합니다.
from langchain_core.tools import BaseTool
assert isinstance(find_window, BaseTool) and find_window.name == 'find_window'
assert find_window.invoke({'minwon': '여권'}) == '3번 창구'
assert find_window.invoke({'minwon': ' 전입신고 '}) == '1번 창구'   # 앞뒤 공백을 다듬었는가
_miss = find_window.invoke({'minwon': '없는민원'})
assert _miss != '없음' and all(w in _miss for w in ['여권', '전입신고', '등본 발급']), \
    '없는 민원에는 고를 수 있는 이름을 알려 주는 안내 문장을 돌려주세요'
# 설명이 판단 근거를 주는가 - 두루뭉술한 find 의 '찾는다.' 는 여기서 전부 걸립니다.
assert '창구' in find_window.description, "docstring 에 '창구' 가 들어가야 합니다"
assert all(w in find_window.description for w in ['여권', '전입신고', '등본 발급']), \
    'docstring 에 고를 수 있는 민원 이름 세 개를 적으세요'
# 에이전트를 실제로 돌렸는지 - 어느 도구가 불렸는지는 위 출력으로 눈으로 확인하세요.
assert any(m.content == '여권 발급하려면 어디로 가야 하나요?' for m in win_result['messages'])
assert any(isinstance(m, AIMessage) and m.response_metadata
           for m in win_result['messages'])
print('✅ 통과!')

**해설**: `find` 와 `find_window` 는 **하는 일이 똑같습니다.** 표를 찾아 창구를 돌려줄 뿐이지요. 달라진 것은 모델이 읽는 **이름과 설명**뿐인데, 그것이 곧 라우팅의 전부입니다 — 모델에게는 함수 본문이 보이지 않고 이름·설명·인자 명세만 보이기 때문입니다.

그래서 docstring 에 적을 것이 정해집니다. **언제 쓰는 도구인지**(창구 안내)와 **무엇을 넣을 수 있는지**(여권·전입신고·등본 발급)입니다. 뒤엣것이 없으면 모델이 '여권 발급하려면...' 에서 인자를 어떻게 잘라 넣을지 몰라 표에 없는 표면형을 넘기게 됩니다.

**흔한 실수**: 없는 값에 `'없음'` 을 돌려주는 것입니다. 모델은 그것을 "정말로 없다" 로 읽고 사용자에게 그대로 전합니다(교안 5절의 **조용한 실패**). LV3 에서 이 증상을 기록으로 진단해 고치게 됩니다.

---
## 8. 문서 경계를 넘지 않게 앞뒤 조각 번호 고르기
**배경**: 검색은 **조각 하나**를 집어 오지만, 모델에게 넘길 때는 그 **앞뒤 조각까지** 붙여야 문맥이 이어집니다. 그러려면 먼저 "어떤 번호의 조각을 가져올지" 를 정해야 하지요. 이 문제는 그 **번호 고르기**만 떼어 연습합니다(실제로 꺼내 오는 일은 LV2 에서 합니다).

**요구사항**: 함수 **`neighbor_numbers(chunk_no, chunk_total, window=1)`** 를 만드세요. 가져올 조각 번호를 **작은 수부터 정렬된 리스트**로 돌려줍니다.

- 조각 번호는 **0 부터** 시작하고, 그 문서의 마지막 번호는 `chunk_total - 1` 입니다.
- 자기 자신(`chunk_no`)도 **포함**합니다.
- 앞뒤로 `window` 개씩 가져오되, **0 보다 작거나 `chunk_total` 이상인 번호는 버립니다** (다른 문서로 넘어가면 안 되니까요).

**예시**

| 입력 | 결과 | 왜 |
|---|---|---|
| `neighbor_numbers(2, 5)` | `[1, 2, 3]` | 앞뒤 한 개씩 |
| `neighbor_numbers(0, 5)` | `[0, 1]` | 앞이 없다 |
| `neighbor_numbers(4, 5)` | `[3, 4]` | 뒤가 없다 |
| `neighbor_numbers(2, 5, window=2)` | `[0, 1, 2, 3, 4]` | 창을 넓히면 더 가져온다 |
| `neighbor_numbers(0, 1)` | `[0]` | 조각이 하나뿐이면 자기 자신만 |

<details><summary>힌트</summary>

```text
접근방법:
- chunk_no 를 가운데 두고 window 만큼 앞뒤로 벌린 범위를 만든 뒤, 문서 밖 번호를 걸러 낸다.

세부구현:
1. 시작은 chunk_no 에서 window 를 뺀 값, 끝은 chunk_no 에 window 를 더한 값이다.
2. 그 범위의 번호를 하나씩 보면서 0 이상이고 chunk_total 보다 작은 것만 남긴다.
3. 남은 번호를 작은 수부터 담은 리스트를 반환한다.
```

</details>

In [ ]:
# 문서 밖으로 나가는 번호를 걸러 내는 것이 이 함수의 전부다
def neighbor_numbers(chunk_no, chunk_total, window=1):
    """가져올 조각 번호를 작은 수부터 정렬해 돌려준다(문서 경계를 넘지 않는다)."""
    return [n for n in range(chunk_no - window, chunk_no + window + 1)
            if 0 <= n < chunk_total]


print(neighbor_numbers(2, 5), neighbor_numbers(0, 5), neighbor_numbers(4, 5))

In [ ]:
# [자가채점]
assert neighbor_numbers(2, 5) == [1, 2, 3]
assert neighbor_numbers(0, 5) == [0, 1], '앞이 없으면 앞 번호는 빼야 합니다'
assert neighbor_numbers(4, 5) == [3, 4], '마지막 번호는 chunk_total - 1 입니다'
assert neighbor_numbers(2, 5, window=2) == [0, 1, 2, 3, 4]
assert neighbor_numbers(0, 1) == [0], '조각이 하나뿐이면 자기 자신만 남습니다'
# 지문에 없는 입력으로 한 번 더 - 값을 손으로 적어 둔 답안은 여기서 걸립니다.
assert neighbor_numbers(7, 20, window=3) == [4, 5, 6, 7, 8, 9, 10]
assert neighbor_numbers(1, 3, window=5) == [0, 1, 2], '창이 문서보다 넓어도 문서 안에서 멈춥니다'
print('✅ 통과!')

**해설**: 범위를 만든 뒤 **문서 밖 번호를 거르는** 한 줄이 전부입니다. `0 <= n < chunk_total` 이 두 경계를 한꺼번에 막아 줍니다.

**흔한 실수 둘**. **1)** `range(chunk_no - window, chunk_no + window)` 로 적어 **뒤쪽 한 개를 빠뜨리는** 것 — 파이썬 `range` 는 끝을 포함하지 않으므로 `+ window + 1` 이어야 합니다. **2)** 경계를 `max(0, ...)` 로만 막고 뒤쪽(`chunk_total`)을 안 막는 것 — 그러면 **없는 번호**를 달라고 하게 되고, 실제 검색에서는 조용히 빈 결과가 섞입니다.

이 번호로 컬렉션에서 조각을 실제로 꺼내 오는 일은 **LV2 1번**에서 합니다. 그때 "꺼낸 순서는 보장되지 않는다" 는 함정이 하나 더 나옵니다.

## 9. (서술형) 메시지 기록을 ReAct 세 단계에 대응시키기
**배경**: `create_agent` 의 메시지 기록은 ReAct 의 **생각·행동·관찰**과 정확히 대응됩니다.

**요구사항**: 아래 기록의 각 메시지가 ReAct 세 단계 중 **무엇에 해당하는지**, 그리고 **옛 방식**(글에 형식을 부탁하고 뜯어 읽던 방식)에서 사람이 하던 어떤 일을 `create_agent` 가 대신하는지 2~3문장으로 서술하세요.

```
HumanMessage:  '주민등록등본 2장 수수료 알려줘'
AIMessage:     tool_calls=[{'name':'lookup_fee','args':{'document':'주민등록등본','count':2}}]
ToolMessage:   '800'
AIMessage:     '주민등록등본 2장의 발급 수수료는 800원입니다.'
```

아래 셀에 답을 적으세요(정답 노트북의 모범 서술과 비교).

**모범 서술**:

- 첫 `AIMessage` 의 `tool_calls` 는 ReAct 의 **행동(Acting)** 입니다 — 어떤 도구를 어떤 입력으로 부를지 정한 것으로, 옛 방식의 `행동:`/`행동입력:` 줄에 해당합니다.
- `ToolMessage` 는 **관찰(Observation)** 입니다 — 도구가 돌려준 결과(`'800'`)로, 옛 방식에서 `관찰:` 로 붙이던 것입니다.
- 마지막 `AIMessage` 는 관찰을 본 뒤의 **생각+최종 답**입니다.
- 옛 방식에서 사람이 **글에서 형식을 찾아 뽑고** 반복까지 직접 짜던 일을, `create_agent` 는 도구 호출을 **구조화**해 받고 루프를 **자동으로** 돌려 대신해 줍니다.

---
수고했어요! LV1 에서 **외부 API 도구·기록 구조 읽기·도구 만들기·설명이 좌우하는 라우팅·조각 이웃 고르기**를 하나씩 익혔습니다. LV2 에서는 이것들을 **조합**해 앞뒤 문맥을 붙이는 검색 도구를 만들고, 여러 도구를 라우팅하는 에이전트를 만듭니다.